# `lineage-return` — pre-merge test notebook

Branch: `lineage-return` &nbsp;|&nbsp; gate **D2** (checkpoint §D2 / §B / §H)

**What this branch adds:** a per-lineage **net metabolic return** reducer on the
opt-in lineage field. Net return of a lineage (keyed by full-genome FNV-1a hash) =
cumulative mouthful eaten by its members - cumulative tax paid by its members.

- C: per-cell life accumulators (`lin_intake_cell` / `lin_tax_cell`) filled on the
  cheap hot path; a persistent genome-hash-keyed reducer table folded from them at
  cell death and at export. Zero-cost / no-op when lineage is **disabled** (default).
- Python: `sim.get_lineage_return()` (structured array: hash, intake, tax, net,
  births, pop, first_step, last_step), `sim.lineage_return_totals()`,
  `sim.get_lineage_intake()` / `sim.get_lineage_tax()` (per-cell).

**Run all cells top-to-bottom; every cell prints `PASS` or raises.**


In [1]:
import os, sys
import numpy as np

ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, 'C', 'evoca.h')):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, os.path.join(ROOT, 'python'))
from evoca_py import EvoCA, make_gol_lut

sim = EvoCA(); sim.init(16, lineage=True)
print('repo root   :', ROOT)
print('has reducer :', hasattr(sim, 'get_lineage_return'))
assert hasattr(sim, 'get_lineage_return'), 'reducer accessor missing - wrong branch / stale dylib'
assert sim.lineage_enabled() is True
print('PASS - setup')

repo root   : /private/tmp/evoca_restructure
has reducer : True
PASS - setup


In [2]:
# 1. Default OFF is a genuine no-op: empty reducer, zero totals, zero per-cell.
s = EvoCA(); s.init(16, food_inc=0.1, m_scale=0.5, tax=0.02, lineage=False)
s.set_lut_all(make_gol_lut()); s.set_egenome_all(0); s.set_alive_all()
s.set_v(np.zeros((16,16), np.uint8)); s.set_F_all(1.0); s.set_f_all(0.5)
for _ in range(4): s.step()
print('rows =', s.get_lineage_return().size, ' totals =', s.lineage_return_totals())
assert s.get_lineage_return().size == 0
assert s.lineage_return_totals() == (0.0, 0.0)
assert not s.get_lineage_intake().any() and not s.get_lineage_tax().any()
s.free()
print('PASS - lineage OFF records nothing')

rows = 0  totals = (0.0, 0.0)
PASS - lineage OFF records nothing


In [3]:
# 2. Zero perturbation: lineage ON vs OFF (same seed/setup) -> identical CA + alive.
def run(lineage):
    s = EvoCA(); s.init(20, food_inc=0.1, m_scale=0.5, tax=0.02,
                        mu_lut=0.002, mu_egenome=0.02, lineage=lineage)
    s.set_seed(4242)
    s.set_lut_all(make_gol_lut()); s.set_egenome_all(0); s.set_alive_all()
    s.set_v(np.zeros((20,20), np.uint8)); s.set_F_all(1.0); s.set_f_all(0.5)
    for _ in range(12): s.step()
    v, a = s.get_v().copy(), s.get_alive().copy(); s.free(); return v, a
v_off, a_off = run(False); v_on, a_on = run(True)
assert np.array_equal(v_off, v_on) and np.array_equal(a_off, a_on)
print('PASS - enabling the reducer does not perturb dynamics')

PASS - enabling the reducer does not perturb dynamics


In [4]:
# 3. Short evolving run: eat + tax + mutation, with births AND deaths.
N, K = 32, 40
sim = EvoCA(); sim.init(N, food_inc=0.08, m_scale=0.5, tax=0.02,
                        mu_lut=0.003, mu_egenome=0.03, lineage=True)
sim.set_seed(7)
sim.set_lut_all(make_gol_lut()); sim.set_egenome_all(0); sim.set_alive_all()
sim.set_v(np.zeros((N, N), np.uint8)); sim.set_F_all(1.0); sim.set_f_all(0.5)
net_traj = []
for t in range(K):
    sim.step()
    sim.lineage_return_update()          # keep live intake/tax + extinction step current
    red = sim.get_lineage_return(update=False)
    net_traj.append(red['net'].sum())    # total net return across all lineages
print('steps run          :', K)
print('distinct lineages  :', red.size)
print('alive at end       :', int(sim.get_alive().sum()), '/', N*N)
assert red.size >= 2, 'mutation should have produced multiple genome lineages'
print('PASS - reducer populated over a run')

steps run          : 40
distinct lineages  : 2823
alive at end       : 1024 / 1024
PASS - reducer populated over a run


In [5]:
# 4. Print a few lineage net-return values (top by |net|).
red = sim.get_lineage_return()
order = np.argsort(-np.abs(red['net']))
print(f"{'hash':>10} {'intake':>10} {'tax':>10} {'net':>10} {'births':>7} {'pop':>4} {'first':>5} {'last':>5}")
for r in red[order][:8]:
    print(f"0x{r['hash']:08x} {r['intake']:10.4f} {r['tax']:10.4f} {r['net']:10.4f} "
          f"{int(r['births']):7d} {int(r['pop']):4d} {int(r['first_step']):5d} {int(r['last_step']):5d}")
print('PASS - per-lineage net-return values printed')

      hash     intake        tax        net  births  pop first  last
0x3dfe0355  1469.2290   211.6200  1257.6090     970   63     1    40
0x407013d0   148.9227    31.8600   117.0627     278   22     2    40
0xbe090c5d    35.7475     8.1200    27.6275      67    7     2    40
0xfdfe0355    27.0278     6.1000    20.9278      35    4     2    40
0xbdfe0355    25.5812     5.6200    19.9612      48    2     2    40
0xf3fe0355    12.5025     2.8400     9.6625      15    5     2    40
0xd3fe0355    11.4050     2.4400     8.9650      16    2     2    40
0x577e0355    10.6775     2.4800     8.1975      11    3     2    40
PASS - per-lineage net-return values printed


In [6]:
# 5. SANITY: intake-tax accounting. The per-lineage decomposition must be
#    complete (columns sum to the independent run-cumulative totals) and
#    net == intake - tax for every lineage.
intake_total, tax_total = sim.lineage_return_totals()
si, st = red['intake'].sum(), red['tax'].sum()
print(f'intake: reducer-sum {si:.6f}  vs total {intake_total:.6f}')
print(f'tax   : reducer-sum {st:.6f}  vs total {tax_total:.6f}')
assert intake_total > 0 and tax_total > 0
assert abs(si - intake_total) < 1e-6 * intake_total, 'intake decomposition incomplete'
assert abs(st - tax_total)    < 1e-6 * tax_total,    'tax decomposition incomplete'
assert np.allclose(red['net'], red['intake'] - red['tax'])
print('PASS - intake-tax accounting is complete and consistent')

intake: reducer-sum 4128.208174  vs total 4128.208174
tax   : reducer-sum 819.199982  vs total 819.199982
PASS - intake-tax accounting is complete and consistent


In [7]:
# 6. Punctuation-style readout: total net-return trajectory (dip-then-climb is the
#    §D signature). We just confirm it is a finite, run-length time series here.
net_traj = np.asarray(net_traj)
print('net-return trajectory length:', net_traj.shape[0])
print('first 5:', np.round(net_traj[:5], 3))
print('last  5:', np.round(net_traj[-5:], 3))
assert net_traj.shape[0] == K and np.isfinite(net_traj).all()
sim.free()
print('PASS - net-return trajectory available for the punctuation detector')

net-return trajectory length: 40
first 5: [ 491.52  512.    808.96  988.16 1024.  ]
last  5: [3070.685 3071.999 3163.28  3239.55  3309.008]
PASS - net-return trajectory available for the punctuation detector


---
**All cells PASS** -> D2 reducer is correct (complete intake-tax accounting), opt-in
(no-op when lineage disabled), non-perturbing, and exposes both per-lineage cumulative
net return and its trajectory for the §D punctuation detector and §H adaptive-vs-drift
program.
